# 2026/8/26

# Function Calling 与 FastAPI + LLM 应用

> **今天的学习主线：让大模型从“只会生成文本”扩展到“能够选择工具、调用外部能力，并通过 Web API 对外提供服务”。**

本次内容可以分为两个部分：

1. **Function Calling**：模型根据用户问题判断是否需要调用函数，并生成符合约束的函数参数；
2. **FastAPI + LLM**：将大模型封装成后端接口，通过流式响应构建语文、数学、英语教师助手。

二者组合后，便形成了一个基础的 LLM 应用链路：

$$
\boxed{
\text{用户请求}
\rightarrow
\text{FastAPI 接口}
\rightarrow
\text{LLM 判断与生成}
\rightarrow
\text{工具/外部服务}
\rightarrow
\text{流式返回结果}
}
$$

## 一、Function Calling

### 1. Function Calling 是什么

大模型本质上只负责生成 token，本身不能直接查询天气、访问数据库或执行本地函数。Function Calling 的作用是让模型根据函数描述，输出一个**结构化的工具调用请求**。

> **模型负责决定“调用哪个工具、传入什么参数”，真正执行函数的仍然是我们的 Python 程序。**

例如用户询问巴黎天气时，完整过程不是模型直接得到实时温度，而是：

```text
用户：巴黎今天天气怎么样？
        ↓
模型：需要调用 get_weather(latitude, longitude)
        ↓
程序：解析参数并真正请求天气 API
        ↓
程序：把查询结果作为 tool 消息交还模型
        ↓
模型：将工具结果组织成自然语言回答
```

Function Calling 解决了两个核心问题：

- 将自然语言问题转换为结构化参数；
- 让 LLM 能够连接实时信息和外部系统，而不是只依赖训练数据。

---

### 2. Function Calling 中各角色的职责

| 组件 | 主要职责 |
|---|---|
| 用户 | 用自然语言提出需求 |
| LLM | 理解意图，选择工具并生成参数 |
| Tool Schema | 告诉模型工具名称、功能和参数格式 |
| Python 程序 | 校验参数并执行真实函数 |
| 外部 API | 提供天气、数据库、搜索等真实数据 |
| LLM 第二次调用 | 将原始工具结果整理为易读回答 |

这里最重要的边界是：**LLM 的输出只是“调用建议”，不能直接当作可信指令执行。** 在真实项目中，还需要进行参数校验、权限控制、超时处理和异常处理。

### 3. 定义真实工具：天气查询函数

课堂示例使用 Open-Meteo 获取指定经纬度的当前气温。与直接拼接长 URL 相比，使用 `params` 更便于阅读、扩展和调试。

In [ ]:
import requests


def get_weather(latitude: float, longitude: float) -> float:
    """查询指定经纬度的当前气温，单位为摄氏度。"""
    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": latitude,
            "longitude": longitude,
            "current": "temperature_2m",
        },
        timeout=10,
    )
    response.raise_for_status()
    data = response.json()
    return data["current"]["temperature_2m"]

### 4. 使用 JSON Schema 描述工具

模型并不会读取 Python 函数本身，因此需要额外提供 Tool Schema。Schema 相当于函数的“使用说明书”，其中包含：

- `name`：工具名称，后续需要与本地函数进行映射；
- `description`：工具用途，帮助模型判断什么时候应该调用；
- `parameters`：参数的 JSON Schema；
- `required`：必填参数；
- `additionalProperties: false`：禁止模型生成未定义的额外字段；
- `strict: true`：要求模型严格遵循给定结构。

> 工具描述越明确，模型选错工具或传错参数的概率越低。

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定经纬度的当前气温，单位为摄氏度。",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "纬度，例如巴黎为 48.8566",
                    },
                    "longitude": {
                        "type": "number",
                        "description": "经度，例如巴黎为 2.3522",
                    },
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

### 5. 第一次调用：让模型决定是否使用工具

第一次请求中同时传入 `messages` 和 `tools`。如果模型认为需要查询实时天气，它通常不会立即给出最终答案，而是在 `message.tool_calls` 中返回工具名称和 JSON 参数。

模型生成的调用信息类似：

```json
{
  "name": "get_weather",
  "arguments": {
    "latitude": 48.8566,
    "longitude": 2.3522
  }
}
```

巴黎的经纬度并不是用户明确给出的，而是模型根据语义补全的参数。

In [ ]:
import os
from openai import OpenAI


client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

messages = [
    {"role": "user", "content": "What's the weather like in Paris today?"}
]

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=tools,
)

### 6. 执行工具并回填结果

程序需要完成以下步骤：

1. 读取 `tool_calls`；
2. 将字符串形式的 `arguments` 解析为 Python 字典；
3. 根据函数名找到本地函数；
4. 执行函数；
5. 使用原始 `tool_call_id` 将结果回填到消息列表。

`tool_call_id` 用于建立“某次工具请求”和“对应工具结果”之间的关系。当一次回答包含多个工具调用时，这个标识尤其重要。

In [ ]:
import json


available_functions = {
    "get_weather": get_weather,
}

assistant_message = completion.choices[0].message
messages.append(assistant_message)

for tool_call in assistant_message.tool_calls or []:
    function_name = tool_call.function.name
    function_args = json.loads(tool_call.function.arguments)

    if function_name not in available_functions:
        raise ValueError(f"未知工具：{function_name}")

    result = available_functions[function_name](**function_args)

    messages.append(
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(
                {"temperature": result, "unit": "celsius"},
                ensure_ascii=False,
            ),
        }
    )

### 7. 第二次调用：生成最终自然语言回答

工具返回的通常是数值、JSON 或数据库记录，不适合直接展示给用户。因此需要把完整消息历史再次交给模型，让模型理解工具结果并生成最终回答。

In [ ]:
final_completion = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    tools=tools,
)

print(final_completion.choices[0].message.content)

### 8. Function Calling 的关键认识

#### 为什么需要调用两次模型？

第一次调用负责**规划动作**，第二次调用负责**基于观察结果回答**：

$$
\text{Question}
\rightarrow
\text{Tool Call}
\rightarrow
\text{Tool Result}
\rightarrow
\text{Final Answer}
$$

这已经具备 Agent 最基础的“思考—行动—观察—回答”结构。

#### 工具参数可以直接信任吗？

不能。即使开启严格模式，也应在程序侧继续检查：

- 数值是否在合理范围内；
- 用户是否拥有调用该工具的权限；
- 工具名称是否存在于白名单；
- 是否出现重复调用或调用次数过多；
- 外部请求是否超时；
- 工具返回内容是否包含敏感信息。

#### 模型一定会调用工具吗？

不一定。模型会根据问题自行判断。实际程序要同时处理两种情况：

- `tool_calls` 存在：执行工具并继续对话；
- `tool_calls` 为空：直接读取模型的文本回答。

<div style="page-break-after: always;"></div>

## 二、FastAPI + LLM：AI 教师助手

### 1. 项目目标

将大模型能力封装成 Web API，并通过不同接口设置不同的 `system prompt`，得到三个学科教师：

| 接口 | 角色定位 | 主要回答特点 |
|---|---|---|
| `POST /chinese-teacher` | 语文老师 | 文学、语言文字、作文指导，引经据典 |
| `POST /math-teacher` | 数学老师 | 先讲思路，再分步骤推导，必要时提供多种解法 |
| `POST /english-teacher` | 英语老师 | 语法、词汇、阅读与写作，中英文结合 |

请求体统一为：

```json
{
  "question": "你的问题"
}
```

项目的核心思想是：**同一个基础模型，通过不同系统提示词承担不同角色；FastAPI 负责参数校验、路由分发和响应输出。**

### 2. FastAPI 中的重要组件

#### Pydantic 请求模型

```python
class QuestionRequest(BaseModel):
    question: str
```

定义请求模型后，FastAPI 会自动完成：

- 读取 JSON 请求体；
- 检查 `question` 是否存在；
- 校验字段类型；
- 自动生成 OpenAPI 文档。

服务启动后可以访问 `/docs`，直接在 Swagger UI 中测试接口。

#### 路由

```python
@app.post("/math-teacher")
async def math_teacher(request: QuestionRequest):
    ...
```

装饰器将 URL 和 Python 函数关联起来。不同路由可以共享同一个大模型调用函数，只需要传入不同的系统提示词。

#### CORS 中间件

CORS 用于允许浏览器中的前端页面跨域访问后端接口。开发阶段可以放宽来源限制，生产环境应显式配置允许访问的前端域名。

### 3. 流式响应

普通请求需要等待模型生成完整答案后才能返回；流式响应会将生成内容分块发送，让用户更快看到首段文字。

课堂代码中的生成器逻辑可以概括为：

```python
for chunk in completion:
    content = chunk.choices[0].delta.content
    if content:
        yield content
```

其工作方式为：

```text
LLM 生成一个 chunk
        ↓
Python 生成器 yield 一段内容
        ↓
FastAPI 立即发送给客户端
        ↓
继续等待下一个 chunk
```

生成器的优点是无需在内存中保存完整回答，并能降低用户感受到的等待时间。

In [ ]:
import json
import os
from collections.abc import Generator

from openai import OpenAI


def qwen_max_stream(
    prompt: str,
    system_prompt: str = "You are a helpful assistant.",
) -> Generator[str, None, None]:
    client = OpenAI(
        api_key=os.environ["DASHSCOPE_API_KEY"],
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )

    try:
        completion = client.chat.completions.create(
            model="qwen-max-latest",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt},
            ],
            stream=True,
        )

        for chunk in completion:
            if chunk.choices and chunk.choices[0].delta.content:
                payload = {
                    "content": chunk.choices[0].delta.content,
                }
                yield f"data: {json.dumps(payload, ensure_ascii=False)}\n\n"

        yield 'data: {"finished": true}\n\n'

    except Exception as exc:
        payload = {"error": str(exc)}
        yield f"data: {json.dumps(payload, ensure_ascii=False)}\n\n"

### 4. Server-Sent Events（SSE）

课堂项目返回的数据格式为：

```text
data: {"content": "第一段文字"}

data: {"content": "第二段文字"}

data: {"finished": true}
```

每条消息以 `data:` 开头，并以两个换行符结束，这是 SSE 的基本格式。若采用这种格式，响应类型应设置为：

```python
return StreamingResponse(
    qwen_max_stream(request.question, system_prompt),
    media_type="text/event-stream",
    headers={"Cache-Control": "no-cache"},
)
```

> 如果只想返回纯文本流，则可以保留 `text/plain`，但生成器中也应直接 `yield content`，不再包装 `data:`。协议格式与 `Content-Type` 应保持一致。

### 5. System Prompt 与多角色复用

三个教师接口的底层调用逻辑几乎相同，主要区别在系统提示词。System Prompt 的作用是限定：

- 模型扮演的角色；
- 回答范围；
- 语言和表达风格；
- 推理与展示步骤；
- 需要遵守的特殊要求。

例如数学老师强调“先给思路、再展示步骤”，英语老师强调“中英文对照和实用例句”。这说明在不微调模型的情况下，也可以通过提示词完成轻量级的角色定制。

不过，Prompt 只是软约束。对于严格的业务规则，仍需要程序校验、内容审核和权限系统配合。

### 6. 当前项目值得改进的地方

#### 1. API Key 不应写在代码中

密钥写入源码会带来泄露风险，也容易被提交到 Git 仓库。应通过环境变量读取：

```python
api_key = os.environ["DASHSCOPE_API_KEY"]
```

若密钥曾经写入或提交到仓库，应及时在服务商控制台**撤销并重新生成**。

#### 2. SSE 响应类型应匹配

返回 `data: ...\n\n` 时，应使用 `text/event-stream`，而不是 `text/plain`。

#### 3. 避免重复路由代码

三个教师接口的结构高度相似，可以抽取统一函数或使用配置表维护角色提示词，减少复制粘贴。

#### 4. 增加输入限制

可以限制问题的最小、最大长度，避免空问题或过长输入：

```python
from pydantic import BaseModel, Field

class QuestionRequest(BaseModel):
    question: str = Field(min_length=1, max_length=2000)
```

#### 5. 完善异常与日志

真实项目应区分网络超时、鉴权失败、模型限流等异常，并在服务端记录日志。直接将完整异常返回给前端可能泄露内部信息。

#### 6. 收紧 CORS

生产环境应把 `allow_origins` 设置为实际前端域名。若允许携带凭证，则不应简单使用通配符来源。

#### 7. 复用模型客户端

当前每次请求都会重新创建 `OpenAI` 客户端。可以在应用启动时创建一次并复用，减少重复初始化。

## 三、Function Calling 与 FastAPI 的结合

今天的两部分内容可以进一步组合：FastAPI 接收用户请求，LLM 判断是否调用天气、搜索或数据库工具，程序执行工具后，再把最终结果流式返回。

```text
前端
  │ HTTP 请求
  ▼
FastAPI 路由
  │ messages + tools
  ▼
LLM 第一次调用
  │ tool_calls
  ▼
工具分发器 ──────► 天气 API / 数据库 / 搜索服务
  │ tool result
  ▼
LLM 第二次调用
  │ 自然语言答案
  ▼
StreamingResponse
  │ SSE 数据流
  ▼
前端逐段展示
```

在这个结构中：

- FastAPI 是应用入口和传输层；
- LLM 是意图理解与语言生成层；
- Function Calling 是 LLM 与外部工具之间的协议；
- 本地函数和第三方 API 是能力执行层；
- SSE 是用户体验层面的流式传输方案。

这也是许多基础 Agent 服务的雏形。

## 四、今日总结

1. **Function Calling 不等于函数自动执行**：模型只生成工具名称和参数，程序负责真正执行；
2. **工具定义依赖 JSON Schema**：清晰的名称、描述和参数约束能够提高调用准确率；
3. **一次完整工具调用通常包含两次模型请求**：第一次决定工具，第二次结合工具结果生成最终答案；
4. **消息历史不能丢失**：需要保留 assistant 的工具调用消息，并使用相同的 `tool_call_id` 回填结果；
5. **FastAPI 可以将 LLM 封装为标准 Web API**：Pydantic 负责输入校验，路由负责角色分发；
6. **流式输出依赖生成器**：模型每生成一段，后端就可以立即向客户端发送一段；
7. **SSE 的格式与响应类型要一致**：使用 `data: ...\n\n` 时，应设置 `text/event-stream`；
8. **密钥必须与源码分离**：使用环境变量管理 API Key，并避免将敏感信息提交到仓库；
9. **Prompt 可以快速塑造角色，但不能替代程序约束**：安全、权限和业务规则仍应由代码保证；
10. **Function Calling + FastAPI 构成了基础 Agent 后端**：模型理解需求，工具执行动作，Web 服务负责连接用户。

> **今天真正打通的是“大模型如何接入应用”的完整思路：模型不再只是聊天窗口，而是后端系统中的一个推理与调度组件。**

<div style="page-break-after: always;"></div>

# 2026/8/28

## 三、FastAPI 基础到异步中间件

> **今天的学习重点：从最小可运行的 API 开始，逐步掌握路由、参数校验、响应模型、异步并发和中间件。**

### 1. 创建应用与启动服务器

FastAPI 应用的核心是 `FastAPI()` 实例，路由通过装饰器绑定到 Python 函数；Uvicorn 负责运行 ASGI 应用。开发阶段使用 `--reload` 可以在代码修改后自动重启。FastAPI 还会自动生成 `/docs` 和 `/redoc` 文档页面。

```python
from fastapi import FastAPI

app = FastAPI(title="FastAPI 学习项目")

@app.get("/")
async def read_root():
    return {"message": "Hello FastAPI!"}

# uvicorn chapter1_basics.main:app --reload --port 8001
```

### 2. 路由、路径参数与查询参数

路径参数描述资源位置，查询参数用于筛选、分页等附加条件。FastAPI 根据类型注解自动转换和校验参数；没有默认值的参数是必填参数，缺失或类型错误时会返回 422。`Enum` 可以限制参数只能取指定值。

```python
@app.get("/items/{item_id}")
async def read_item(item_id: int, q: str | None = None):
    return {"item_id": item_id, "q": q}
```

### 3. Pydantic 数据模型与校验

使用 `BaseModel` 定义请求体，使用 `Field` 描述长度、数值范围和正则约束。复杂业务可以组合嵌套模型、列表和枚举，并通过自定义验证器检查作者姓名、年份、订单重复项等规则。校验失败会在进入业务逻辑前被拦截。

```python
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(..., min_length=1, max_length=100)
    price: float = Field(..., gt=0)
    discount: float | None = Field(None, ge=0, le=100)
```

对于资源不存在等业务错误，应使用 `HTTPException` 返回明确的状态码和错误信息。

### 4. 响应模型与安全过滤

请求模型和响应模型应分开设计。通过 `response_model` 可以自动过滤数据库对象中的敏感字段，例如不把 `password` 返回给客户端，同时还能约束响应结构并生成接口文档。

### 5. 同步、异步与并发

`async def` 适合等待数据库、HTTP 等 I/O 操作；等待期间事件循环可以处理其他请求。多个互不依赖的异步任务可用 `asyncio.gather` 并发执行：

$$
T_{seq}=T_1+T_2+T_3, \qquad T_{concurrent}\approx\max(T_1,T_2,T_3)
$$

示例中三个任务耗时分别为 1、1.5、0.5 秒，顺序执行约需 3 秒，并发执行约需 1.5 秒。需要注意：声明为 `async` 并不自动产生并发，必须通过任务调度和 `gather` 等方式让任务重叠执行。

### 6. HTTP 中间件

中间件包裹所有请求，在路由前后执行，适合放置日志、耗时统计、安全响应头和限流等横切逻辑。示例中的计时中间件通过 `call_next` 调用下一个处理器，并把处理耗时写入 `X-Process-Time` 响应头。内存字典限流只适合学习演示，生产环境应使用 Redis 等共享存储。

### 7. CORS、GZip 与 WebSocket

CORS 用于允许指定前端跨域访问；GZipMiddleware 可以压缩较大的响应，减少网络传输量；WebSocket 提供客户端与服务器之间的持续双向连接，适合实时消息和交互式应用。生产环境应收紧 CORS 来源，不应随意使用 `*`。

### 8. 与 LLM 应用结合

FastAPI 可以作为 LLM 应用的 Web 层：Pydantic 校验用户问题，路由选择不同角色，异步代码等待模型响应，中间件负责日志与限流，SSE 或 WebSocket 负责流式传输。整体链路为：

```text
用户请求 → FastAPI 路由 → 参数校验 → LLM 调用 → 流式响应 → 前端展示
```

### 9. 今日总结

1. FastAPI 使用类型注解和 Pydantic 将请求、校验与文档统一起来；
2. 路径参数、查询参数和请求体分别对应不同的输入场景；
3. `response_model` 可以约束输出并过滤敏感字段；
4. 异步适合 I/O，`asyncio.gather` 可以让独立任务并发执行；
5. 中间件适合实现日志、计时、限流和安全头等通用能力；
6. FastAPI 为 Function Calling 和 LLM 流式应用提供了清晰、可校验、可扩展的后端基础。

> **今天的核心收获：FastAPI 把 Python 函数、数据模型和异步能力组织成了可访问的 Web API，是将 LLM 能力落地为应用服务的重要基础。**